# 🚦 Traffic Demand Prediction — Advanced Pipeline

**Target:** Maximize R² score → aiming for **92+**

**Key improvements over v1 (90.79):**
1. ✅ **OOF target encoding** — eliminates leakage from v1
2. ✅ **Advanced feature engineering** — cyclical time, interaction features, geo aggregations
3. ✅ **Three models** — CatBoost + LightGBM + XGBoost
4. ✅ **Optuna hyperparameter tuning** — 50 trials per model
5. ✅ **KFold (5-fold)** training — no train_test_split
6. ✅ **Stacking ensemble** — meta-learner on OOF predictions
7. ✅ **Log-target experiment** — test log1p(y) transformation
8. ✅ **Feature importance + pruning**
9. ✅ **Error analysis** — identify weak spots and add features

## 1. Setup & Imports

In [ ]:
!pip install catboost lightgbm xgboost optuna geohash2 -q

In [ ]:
import pandas as pd
import numpy as np
import geohash2
import optuna
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge, ElasticNet, LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

SEED    = 42
N_FOLDS = 5
print('✅ All imports successful!')

## 2. Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
sample_submission = pd.read_csv('sample_submission.csv')

print(f'Train shape: {train.shape}')   # (77299, 11)
print(f'Test shape:  {test.shape}')    # (41778, 10)
print(f'\nMissing in train:\n{train.isnull().sum()}')
train.head()

## 3. Feature Engineering

**Key insight from data analysis:**
- `day` has only 2 values: 48 (train-only) and 49 (both train rows 0:00–2:45 AND test rows 2:15–13:45)
- `timestamp` is a 15-min interval string like `'7:30'`
- `geohash` is 6-char precision (~1.2 km×0.6 km cells), 1249 unique locations
- Target `demand` is in [0, 1]

> ⚠️ **Leakage fix:** v1 computed `geo_mean_demand` and `hour_mean_demand` on ALL training data before validation. This inflates CV scores. We replace this with **Out-Of-Fold (OOF) target encoding** below.

In [ ]:
def feature_engineer(df, is_train=True, train_ref=None):
    """
    Core feature engineering. Pass train_ref=train when processing test,
    so geo aggregations are computed only from training data.
    """
    df = df.copy()

    # ── Time features ────────────────────────────────────────────────────────
    df['hour']        = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['minute']      = df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df['quarter_hour']= df['hour'] * 4 + df['minute'] // 15  # 0–95
    df['time_of_day'] = df['hour'] + df['minute'] / 60.0     # continuous 0–24

    # Cyclical encoding — hour
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    # Cyclical encoding — quarter-hour (finer granularity)
    df['qh_sin']   = np.sin(2 * np.pi * df['quarter_hour'] / 96)
    df['qh_cos']   = np.cos(2 * np.pi * df['quarter_hour'] / 96)

    # Business-knowledge flags
    df['is_peak']      = df['hour'].apply(lambda h: 1 if (7<=h<=9 or 17<=h<=19) else 0)
    df['is_night']     = df['hour'].apply(lambda h: 1 if (h<=5 or h>=22) else 0)
    df['is_morning']   = df['hour'].apply(lambda h: 1 if 5<=h<=12 else 0)
    df['is_afternoon'] = df['hour'].apply(lambda h: 1 if 12<=h<=18 else 0)

    # ── Geohash → Lat/Lon ────────────────────────────────────────────────────
    decoded   = df['geohash'].apply(lambda g: geohash2.decode(g))
    df['lat'] = decoded.apply(lambda x: float(x[0]))
    df['lon'] = decoded.apply(lambda x: float(x[1]))

    # ── Missing-value indicators ──────────────────────────────────────────────
    df['temperature_missing'] = df['Temperature'].isna().astype(int)
    df['weather_missing']     = df['Weather'].isna().astype(int)
    df['roadtype_missing']    = df['RoadType'].isna().astype(int)

    # ── Fill missing values ───────────────────────────────────────────────────
    ref = train_ref if train_ref is not None else df
    df['RoadType']    = df['RoadType'].fillna('Unknown')
    df['Weather']     = df['Weather'].fillna('Unknown')
    df['Temperature'] = df['Temperature'].fillna(ref['Temperature'].median())

    # ── Binary encodings ──────────────────────────────────────────────────────
    df['LargeVehicles_enc'] = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['Landmarks_enc']     = (df['Landmarks'] == 'Yes').astype(int)

    # ── Interaction features (string combos — CatBoost handles natively) ──────
    df['road_weather']    = df['RoadType'] + '_' + df['Weather']
    df['road_landmark']   = df['RoadType'] + '_' + df['Landmarks']
    df['road_vehicle']    = df['RoadType'] + '_' + df['LargeVehicles']
    df['weather_landmark']= df['Weather']  + '_' + df['Landmarks']
    df['road_hour_bin']   = df['RoadType'] + '_' + (df['hour'] // 6).astype(str)  # 4 time-of-day buckets

    # ── Geo frequency ────────────────────────────────────────────────────────
    geo_freq = ref['geohash'].value_counts() / len(ref)
    df['geo_frequency'] = df['geohash'].map(geo_freq).fillna(0)

    # ── Geo aggregations from training data ───────────────────────────────────
    geo_stats = ref.groupby('geohash').agg(
        geo_mean_temp=('Temperature', 'mean'),
        geo_mean_lanes=('NumberofLanes', 'mean'),
        geo_std_temp=('Temperature', 'std')
    )
    df = df.merge(geo_stats, on='geohash', how='left')
    df['geo_mean_temp']  = df['geo_mean_temp'].fillna(ref['Temperature'].mean())
    df['geo_mean_lanes'] = df['geo_mean_lanes'].fillna(ref['NumberofLanes'].mean())
    df['geo_std_temp']   = df['geo_std_temp'].fillna(0)

    # ── Lanes × Temperature interaction ──────────────────────────────────────
    df['lanes_x_temp'] = df['NumberofLanes'] * df['Temperature']

    return df


train = feature_engineer(train, is_train=True)
test  = feature_engineer(test,  is_train=False, train_ref=train)

print(f'✅ Feature engineering done!')
print(f'Train shape: {train.shape}, Test shape: {test.shape}')

## 4. Label-encode categorical interactions (for LGB / XGB)

CatBoost handles strings natively, but LGB and XGB need numeric input.
We fit encoders on `train ∪ test` to avoid unseen-label errors.

In [ ]:
CAT_COLS_STR = ['RoadType', 'Weather', 'road_weather', 'road_landmark',
                'road_vehicle', 'weather_landmark', 'road_hour_bin']

label_encoders = {}
for col in CAT_COLS_STR:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], ignore_index=True)
    le.fit(combined)
    label_encoders[col] = le
    train[col + '_enc'] = le.transform(train[col])
    test[col  + '_enc'] = le.transform(test[col])

print('✅ Label encoding done')

## 5. OOF Target Encoding (leakage-free)

**Why this matters:**  
In v1, `geo_mean_demand` and `hour_mean_demand` were computed on the **whole** training set, then used as features during cross-validation. This leaks target information into the validation folds, inflating the CV score.

**Fix:** For each fold, compute mean demand per geohash/hour using only the **training** portion of that fold.

In [ ]:
y = train['demand'].values
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Columns to target-encode
TE_COLS = ['geohash', 'hour', 'quarter_hour', 'RoadType', 'Weather']

# Storage for OOF encoded values
oof_te  = {col: np.zeros(len(train))  for col in TE_COLS}
test_te = {col: np.zeros(len(test))   for col in TE_COLS}

global_mean = y.mean()

for fold, (tr_idx, val_idx) in enumerate(kf.split(np.arange(len(train)))):
    y_tr = y[tr_idx]
    for col in TE_COLS:
        col_tr  = train[col].iloc[tr_idx]
        col_val = train[col].iloc[val_idx]
        # Mean target per category, computed only from fold's training split
        mapping = pd.Series(y_tr, index=col_tr.values).groupby(level=0).mean()
        oof_te[col][val_idx] = col_val.map(mapping).fillna(global_mean).values
        # Accumulate test predictions across folds
        test_te[col] += test[col].map(mapping).fillna(global_mean).values / N_FOLDS

for col in TE_COLS:
    train[f'{col}_te'] = oof_te[col]
    test[f'{col}_te']  = test_te[col]

print('✅ OOF target encoding done — no leakage!')
print(f'Added {len(TE_COLS)} OOF target-encoded features.')

## 6. Log-Target Experiment

Test whether `log1p(demand)` as the training target improves CV R².

In [ ]:
y_log = np.log1p(y)  # log1p because demand can be near-zero

print(f'Original demand: min={y.min():.4f}, max={y.max():.4f}, skew={pd.Series(y).skew():.2f}')
print(f'log1p demand:    min={y_log.min():.4f}, max={y_log.max():.4f}, skew={pd.Series(y_log).skew():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist(y, bins=60, color='steelblue'); axes[0].set_title('Original demand')
axes[1].hist(y_log, bins=60, color='coral');  axes[1].set_title('log1p(demand)')
plt.tight_layout(); plt.show()

In [ ]:
# We'll run a quick 2-fold test to decide which target to use
from lightgbm import LGBMRegressor

PROBE_FEATURES = ['hour_te', 'geohash_te', 'quarter_hour_te', 'lat', 'lon',
                  'NumberofLanes', 'Temperature', 'LargeVehicles_enc', 'Landmarks_enc',
                  'hour_sin', 'hour_cos', 'geo_frequency']

kf2 = KFold(n_splits=2, shuffle=True, random_state=SEED)
X_probe = train[PROBE_FEATURES].values

scores_orig, scores_log = [], []
for tr_idx, val_idx in kf2.split(X_probe):
    probe_model = LGBMRegressor(n_estimators=300, learning_rate=0.05, verbose=-1, random_state=SEED)
    
    # Original target
    probe_model.fit(X_probe[tr_idx], y[tr_idx])
    scores_orig.append(r2_score(y[val_idx], probe_model.predict(X_probe[val_idx])))
    
    # Log target
    probe_model.fit(X_probe[tr_idx], y_log[tr_idx])
    preds_log = np.expm1(probe_model.predict(X_probe[val_idx]))
    scores_log.append(r2_score(y[val_idx], preds_log))

print(f'Original target CV R²: {np.mean(scores_orig)*100:.2f}')
print(f'log1p target CV R²:    {np.mean(scores_log)*100:.2f}')
USE_LOG = np.mean(scores_log) > np.mean(scores_orig)
print(f'\n→ Using {"LOG" if USE_LOG else "ORIGINAL"} target for training')

In [ ]:
y_train = y_log if USE_LOG else y

## 7. Define Feature Sets

In [ ]:
# Numeric features for LGB and XGB
FEATURES_NUMERIC = [
    # Time
    'hour', 'minute', 'quarter_hour', 'time_of_day',
    'hour_sin', 'hour_cos', 'qh_sin', 'qh_cos',
    'is_peak', 'is_night', 'is_morning', 'is_afternoon', 'day',
    # Location
    'lat', 'lon', 'geo_frequency',
    'geo_mean_temp', 'geo_mean_lanes', 'geo_std_temp',
    # Road / conditions
    'NumberofLanes', 'Temperature', 'lanes_x_temp',
    'LargeVehicles_enc', 'Landmarks_enc',
    # Missing indicators
    'temperature_missing', 'weather_missing', 'roadtype_missing',
    # Label-encoded categoricals
    'RoadType_enc', 'Weather_enc',
    'road_weather_enc', 'road_landmark_enc', 'road_vehicle_enc',
    'weather_landmark_enc', 'road_hour_bin_enc',
    # OOF target encodings
    'geohash_te', 'hour_te', 'quarter_hour_te', 'RoadType_te', 'Weather_te'
]

# For CatBoost: use raw string categoricals instead of label-encoded ones
FEATURES_CAT_BASE = [
    'hour', 'minute', 'quarter_hour', 'time_of_day',
    'hour_sin', 'hour_cos', 'qh_sin', 'qh_cos',
    'is_peak', 'is_night', 'is_morning', 'is_afternoon', 'day',
    'lat', 'lon', 'geo_frequency',
    'geo_mean_temp', 'geo_mean_lanes', 'geo_std_temp',
    'NumberofLanes', 'Temperature', 'lanes_x_temp',
    'LargeVehicles_enc', 'Landmarks_enc',
    'temperature_missing', 'weather_missing', 'roadtype_missing',
    'geohash_te', 'hour_te', 'quarter_hour_te', 'RoadType_te', 'Weather_te',
    # String categoricals (CatBoost handles natively)
    'RoadType', 'Weather', 'road_weather', 'road_landmark',
    'road_vehicle', 'weather_landmark', 'road_hour_bin'
]
CAT_FEATURES_CB = ['RoadType', 'Weather', 'road_weather', 'road_landmark',
                   'road_vehicle', 'weather_landmark', 'road_hour_bin']

X_num  = train[FEATURES_NUMERIC].values
X_cb   = train[FEATURES_CAT_BASE]
X_test_num = test[FEATURES_NUMERIC].values
X_test_cb  = test[FEATURES_CAT_BASE]

print(f'Numeric feature count: {len(FEATURES_NUMERIC)}')
print(f'CatBoost feature count: {len(FEATURES_CAT_BASE)}')

## 8. Optuna Hyperparameter Tuning

We tune each model with 50–100 Optuna trials, optimizing 3-fold CV R².
Using 3-fold (not 5) inside Optuna for speed — final training uses 5-fold.

In [ ]:
# ── Shared helper ────────────────────────────────────────────────────────────
def cv_r2(model_fn, X, y, n_splits=3):
    """Fast cross-validated R² for Optuna (3-fold)."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    scores = []
    for tr, val in kf.split(X):
        m = model_fn()
        if hasattr(X, 'iloc'):
            m.fit(X.iloc[tr], y[tr])
            p = m.predict(X.iloc[val])
        else:
            m.fit(X[tr], y[tr])
            p = m.predict(X[val])
        # If log target, inverse before scoring
        if USE_LOG:
            p = np.expm1(p)
            scores.append(r2_score(y_orig[val], p))
        else:
            scores.append(r2_score(y[val], p))
    return np.mean(scores)

y_orig = y  # always keep original y for scoring

In [ ]:
# ── Tune LightGBM ────────────────────────────────────────────────────────────
def objective_lgb(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 63, 255),
        'max_depth':         trial.suggest_int('max_depth', 6, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 60),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1':         trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True),
        'lambda_l2':         trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True),
        'verbose': -1, 'random_state': SEED
    }
    fn = lambda: LGBMRegressor(**params)
    return cv_r2(fn, X_num, y_train)

print('Tuning LightGBM... (50 trials)')
study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgb.optimize(objective_lgb, n_trials=50, show_progress_bar=True)
best_lgb = study_lgb.best_params
print(f'\n✅ Best LGB CV R²: {study_lgb.best_value*100:.2f}')
print(f'Best params: {best_lgb}')

In [ ]:
# ── Tune XGBoost ─────────────────────────────────────────────────────────────
def objective_xgb(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth':       trial.suggest_int('max_depth', 5, 10),
        'subsample':       trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma':           trial.suggest_float('gamma', 0.0, 5.0),
        'min_child_weight':trial.suggest_int('min_child_weight', 1, 20),
        'reg_alpha':       trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':      trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'tree_method': 'hist', 'device': 'cpu',
        'verbosity': 0, 'random_state': SEED
    }
    fn = lambda: XGBRegressor(**params)
    return cv_r2(fn, X_num, y_train)

print('Tuning XGBoost... (50 trials)')
study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)
best_xgb = study_xgb.best_params
print(f'\n✅ Best XGB CV R²: {study_xgb.best_value*100:.2f}')
print(f'Best params: {best_xgb}')

In [ ]:
# ── Tune CatBoost ─────────────────────────────────────────────────────────────
def objective_cb(trial):
    params = {
        'iterations':          trial.suggest_int('iterations', 500, 2000),
        'learning_rate':       trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'depth':               trial.suggest_int('depth', 6, 10),
        'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength':     trial.suggest_float('random_strength', 0.0, 10.0),
        'border_count':        trial.suggest_int('border_count', 32, 128),
        'cat_features': CAT_FEATURES_CB,
        'random_seed': SEED, 'verbose': 0
    }
    fn = lambda: CatBoostRegressor(**params)
    return cv_r2(fn, X_cb, y_train)

print('Tuning CatBoost... (50 trials)')
study_cb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_cb.optimize(objective_cb, n_trials=50, show_progress_bar=True)
best_cb = study_cb.best_params
print(f'\n✅ Best CatBoost CV R²: {study_cb.best_value*100:.2f}')
print(f'Best params: {best_cb}')

## 9. KFold Training — Generate OOF Predictions (Level 1)

All three models trained with 5-fold CV using tuned hyperparameters.
OOF predictions are used as input to the Level-2 stacking meta-model.

In [ ]:
kf5 = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Storage
oof_lgb   = np.zeros(len(train))
oof_xgb   = np.zeros(len(train))
oof_cb    = np.zeros(len(train))
test_lgb  = np.zeros(len(test))
test_xgb  = np.zeros(len(test))
test_cb   = np.zeros(len(test))

fold_scores = {'lgb': [], 'xgb': [], 'cb': []}

print('=' * 60)
print(f'5-Fold Training — LightGBM + XGBoost + CatBoost')
print('=' * 60)

for fold, (tr_idx, val_idx) in enumerate(kf5.split(np.arange(len(train)))):
    print(f'\n── Fold {fold + 1} ──────────────────────')
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    y_orig_val  = y_orig[val_idx]

    # ── LightGBM ─────────────────────────────────────────────────────────────
    lgb_params = {**best_lgb, 'verbose': -1, 'random_state': SEED}
    lgb_m = LGBMRegressor(**lgb_params)
    lgb_m.fit(X_num[tr_idx], y_tr,
              eval_set=[(X_num[val_idx], y_val)],
              callbacks=[optuna.integration.lightgbm.LightGBMPruningCallback] if False else [])
    oof_lgb[val_idx] = lgb_m.predict(X_num[val_idx])
    test_lgb += lgb_m.predict(X_test_num) / N_FOLDS
    raw_lgb = np.expm1(oof_lgb[val_idx]) if USE_LOG else oof_lgb[val_idx]
    s_lgb = r2_score(y_orig_val, raw_lgb)
    fold_scores['lgb'].append(s_lgb)
    print(f'  LGB  R²: {s_lgb * 100:.2f}')

    # ── XGBoost ──────────────────────────────────────────────────────────────
    xgb_params = {**best_xgb, 'tree_method': 'hist', 'device': 'cpu',
                  'verbosity': 0, 'random_state': SEED}
    xgb_m = XGBRegressor(**xgb_params)
    xgb_m.fit(X_num[tr_idx], y_tr,
              eval_set=[(X_num[val_idx], y_val)],
              verbose=False)
    oof_xgb[val_idx] = xgb_m.predict(X_num[val_idx])
    test_xgb += xgb_m.predict(X_test_num) / N_FOLDS
    raw_xgb = np.expm1(oof_xgb[val_idx]) if USE_LOG else oof_xgb[val_idx]
    s_xgb = r2_score(y_orig_val, raw_xgb)
    fold_scores['xgb'].append(s_xgb)
    print(f'  XGB  R²: {s_xgb * 100:.2f}')

    # ── CatBoost ─────────────────────────────────────────────────────────────
    cb_params = {**best_cb, 'cat_features': CAT_FEATURES_CB,
                 'random_seed': SEED, 'verbose': 0}
    cb_m = CatBoostRegressor(**cb_params)
    cb_m.fit(X_cb.iloc[tr_idx], y_tr,
             eval_set=(X_cb.iloc[val_idx], y_val),
             early_stopping_rounds=50)
    oof_cb[val_idx] = cb_m.predict(X_cb.iloc[val_idx])
    test_cb += cb_m.predict(X_test_cb) / N_FOLDS
    raw_cb = np.expm1(oof_cb[val_idx]) if USE_LOG else oof_cb[val_idx]
    s_cb = r2_score(y_orig_val, raw_cb)
    fold_scores['cb'].append(s_cb)
    print(f'  CB   R²: {s_cb * 100:.2f}')

# ── Summary ──────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
# Convert OOF predictions back to original scale if log target was used
oof_lgb_orig = np.expm1(oof_lgb) if USE_LOG else oof_lgb
oof_xgb_orig = np.expm1(oof_xgb) if USE_LOG else oof_xgb
oof_cb_orig  = np.expm1(oof_cb)  if USE_LOG else oof_cb
test_lgb_orig = np.expm1(test_lgb) if USE_LOG else test_lgb
test_xgb_orig = np.expm1(test_xgb) if USE_LOG else test_xgb
test_cb_orig  = np.expm1(test_cb)  if USE_LOG else test_cb

print(f'LGB  OOF R²: {r2_score(y_orig, oof_lgb_orig)*100:.2f}  (mean fold: {np.mean(fold_scores["lgb"])*100:.2f})')
print(f'XGB  OOF R²: {r2_score(y_orig, oof_xgb_orig)*100:.2f}  (mean fold: {np.mean(fold_scores["xgb"])*100:.2f})')
print(f'CB   OOF R²: {r2_score(y_orig, oof_cb_orig)*100:.2f}  (mean fold: {np.mean(fold_scores["cb"])*100:.2f})')

## 10. Feature Importance Analysis

In [ ]:
# Use the last fold's trained models for importance
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# LGB importance
lgb_imp = pd.Series(lgb_m.feature_importances_, index=FEATURES_NUMERIC).sort_values(ascending=True)
lgb_imp.tail(20).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('LightGBM Feature Importance (top 20)')

# XGB importance
xgb_imp = pd.Series(xgb_m.feature_importances_, index=FEATURES_NUMERIC).sort_values(ascending=True)
xgb_imp.tail(20).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('XGBoost Feature Importance (top 20)')

# CatBoost importance
cb_imp = pd.Series(cb_m.get_feature_importance(), index=FEATURES_CAT_BASE).sort_values(ascending=True)
cb_imp.tail(20).plot(kind='barh', ax=axes[2], color='seagreen')
axes[2].set_title('CatBoost Feature Importance (top 20)')

plt.tight_layout()
plt.show()

print('Top 10 features (LGB):')
print(lgb_imp.sort_values(ascending=False).head(10))

In [ ]:
# Identify and drop useless LGB features (importance = 0)
zero_imp_features = lgb_imp[lgb_imp == 0].index.tolist()
print(f'Features with zero LGB importance: {zero_imp_features}')

# Optionally retrain without them (uncomment if many zero-importance features found)
# FEATURES_PRUNED = [f for f in FEATURES_NUMERIC if f not in zero_imp_features]
# print(f'Pruned feature count: {len(FEATURES_PRUNED)} (was {len(FEATURES_NUMERIC)})')
# X_num_pruned = train[FEATURES_PRUNED].values
# X_test_num_pruned = test[FEATURES_PRUNED].values
# ... retrain with X_num_pruned

## 11. Error Analysis

In [ ]:
# Residual analysis using the best single-model OOF predictions
best_single_oof = max(
    [(oof_lgb_orig, 'LGB'), (oof_xgb_orig, 'XGB'), (oof_cb_orig, 'CB')],
    key=lambda t: r2_score(y_orig, t[0])
)
best_oof, best_name = best_single_oof

residuals = y_orig - best_oof
train['residual'] = residuals

print(f'Using {best_name} OOF for error analysis')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Error by hour
hour_err = train.groupby('hour')['residual'].apply(lambda x: np.mean(x**2)**0.5)
axes[0].bar(hour_err.index, hour_err.values, color='steelblue')
axes[0].set_title('RMSE by Hour')
axes[0].set_xlabel('Hour')

# Error by road type
road_err = train.groupby('RoadType')['residual'].apply(lambda x: np.mean(x**2)**0.5)
road_err.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('RMSE by RoadType')

# Error by geohash (top 20 worst)
geo_err = train.groupby('geohash')['residual'].apply(lambda x: np.mean(x**2)**0.5).sort_values(ascending=False)
geo_err.head(20).plot(kind='bar', ax=axes[2], color='seagreen')
axes[2].set_title('Top 20 worst geohashes by RMSE')

plt.tight_layout()
plt.show()

print(f'\nHour with highest error: {hour_err.idxmax()} (RMSE={hour_err.max():.4f})')
print(f'RoadType with highest error: {road_err.idxmax()} (RMSE={road_err.max():.4f})')

train.drop('residual', axis=1, inplace=True)

In [ ]:
# If there's a clear hour or geohash pattern, add targeted features here.
# Example: if highway at peak hour is consistently underpredicted:
train['highway_peak'] = ((train['RoadType'] == 'Highway') & (train['is_peak'] == 1)).astype(int)
test['highway_peak']  = ((test['RoadType']  == 'Highway') & (test['is_peak']  == 1)).astype(int)

# Append to feature sets
if 'highway_peak' not in FEATURES_NUMERIC:
    FEATURES_NUMERIC.append('highway_peak')
    FEATURES_CAT_BASE.append('highway_peak')
    X_num      = train[FEATURES_NUMERIC].values
    X_test_num = test[FEATURES_NUMERIC].values
    X_cb       = train[FEATURES_CAT_BASE]
    X_test_cb  = test[FEATURES_CAT_BASE]
    print('Added highway_peak feature')

## 12. Stacking Ensemble (Level 2)

The meta-model is trained on OOF predictions from all three Level-1 models.
We test Ridge, ElasticNet, and LinearRegression and pick the best.
If stacking underperforms simple weighted blending, we fall back to blending.

In [ ]:
# Build Level-1 OOF matrix (shape: n_train × 3)
L1_oof  = np.column_stack([oof_lgb_orig, oof_xgb_orig, oof_cb_orig])
L1_test = np.column_stack([test_lgb_orig, test_xgb_orig, test_cb_orig])

# ── Test meta-learners ───────────────────────────────────────────────────────
meta_candidates = {
    'Ridge':         Ridge(alpha=1.0),
    'ElasticNet':    ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000),
    'LinearReg':     LinearRegression()
}

meta_scores = {}
for name, meta in meta_candidates.items():
    # OOF evaluation of meta-model
    meta_oof = np.zeros(len(train))
    for tr, val in KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(L1_oof):
        meta.fit(L1_oof[tr], y_orig[tr])
        meta_oof[val] = meta.predict(L1_oof[val])
    s = r2_score(y_orig, meta_oof)
    meta_scores[name] = s
    print(f'{name} meta R²: {s*100:.2f}')

best_meta_name = max(meta_scores, key=meta_scores.get)
best_meta_score = meta_scores[best_meta_name]
print(f'\nBest meta-learner: {best_meta_name} ({best_meta_score*100:.2f})')

In [ ]:
# ── Compare stacking vs weighted blending ────────────────────────────────────
scores_individual = {
    'LGB': r2_score(y_orig, oof_lgb_orig),
    'XGB': r2_score(y_orig, oof_xgb_orig),
    'CB':  r2_score(y_orig, oof_cb_orig)
}

# Simple equal-weight blend
equal_blend = (oof_lgb_orig + oof_xgb_orig + oof_cb_orig) / 3
equal_blend_score = r2_score(y_orig, equal_blend)

# Optimized weighted blend via Optuna
def blend_objective(trial):
    w1 = trial.suggest_float('w_lgb', 0.1, 0.8)
    w2 = trial.suggest_float('w_xgb', 0.1, 0.8)
    w3 = 1.0 - w1 - w2
    if w3 < 0.05 or w3 > 0.8:
        return 0.0
    blend = w1 * oof_lgb_orig + w2 * oof_xgb_orig + w3 * oof_cb_orig
    return r2_score(y_orig, blend)

blend_study = optuna.create_study(direction='maximize')
blend_study.optimize(blend_objective, n_trials=200, show_progress_bar=False)
w_lgb = blend_study.best_params['w_lgb']
w_xgb = blend_study.best_params['w_xgb']
w_cb  = 1.0 - w_lgb - w_xgb
opt_blend = w_lgb * oof_lgb_orig + w_xgb * oof_xgb_orig + w_cb * oof_cb_orig
opt_blend_score = r2_score(y_orig, opt_blend)

print(f'Individual model OOF R²: LGB={scores_individual["LGB"]*100:.2f}  '
      f'XGB={scores_individual["XGB"]*100:.2f}  CB={scores_individual["CB"]*100:.2f}')
print(f'Equal blend OOF R²:      {equal_blend_score*100:.2f}')
print(f'Optimal blend OOF R²:    {opt_blend_score*100:.2f}  (w_lgb={w_lgb:.3f}, w_xgb={w_xgb:.3f}, w_cb={w_cb:.3f})')
print(f'Best stacking OOF R²:    {best_meta_score*100:.2f}  ({best_meta_name})')

In [ ]:
# ── Auto-select best ensemble strategy ──────────────────────────────────────
strategy_scores = {
    'stacking': best_meta_score,
    'opt_blend': opt_blend_score,
    'equal_blend': equal_blend_score
}
best_strategy = max(strategy_scores, key=strategy_scores.get)
print(f'\n🏆 Best ensemble strategy: {best_strategy} (OOF R² = {strategy_scores[best_strategy]*100:.2f})')

# Generate final test predictions
if best_strategy == 'stacking':
    best_meta_model = meta_candidates[best_meta_name]
    best_meta_model.fit(L1_oof, y_orig)
    final_test_preds = best_meta_model.predict(L1_test)
elif best_strategy == 'opt_blend':
    final_test_preds = w_lgb * test_lgb_orig + w_xgb * test_xgb_orig + w_cb * test_cb_orig
else:
    final_test_preds = (test_lgb_orig + test_xgb_orig + test_cb_orig) / 3

# Clip to valid range [0, 1] — demand is bounded
final_test_preds = np.clip(final_test_preds, 0, 1)
print(f'Test predictions: min={final_test_preds.min():.4f}, max={final_test_preds.max():.4f}')

## 13. Final Score Summary

In [ ]:
final_oof = (
    best_meta_model.predict(L1_oof)    if best_strategy == 'stacking'
    else opt_blend                     if best_strategy == 'opt_blend'
    else equal_blend
)
final_oof = np.clip(final_oof, 0, 1)
final_cv_score = r2_score(y_orig, final_oof)

print('=' * 50)
print(f'  V1 Score (single split, leaky):  90.79')
print(f'  V2 Final OOF CV R²:              {final_cv_score * 100:.2f}')
print(f'  Strategy: {best_strategy}')
print('=' * 50)

print(f'\nOptuna best params:')
print(f'  LGB:  {best_lgb}')
print(f'  XGB:  {best_xgb}')
print(f'  CB:   {best_cb}')

## 14. Create Submission File

In [ ]:
submission = pd.DataFrame({
    'Index':  test['Index'],
    'demand': final_test_preds
})

# Sanity checks
assert submission.shape == (41778, 2),          f'Wrong shape: {submission.shape}'
assert not submission.isnull().any().any(),      'Submission has nulls!'
assert list(submission.columns) == ['Index', 'demand'], 'Wrong column names!'
assert submission['demand'].between(0, 1).all(), 'Demand outside [0, 1]!'

submission.to_csv('submission_v2.csv', index=False)
print(f'✅ submission_v2.csv saved — shape: {submission.shape}')
print(f'   Demand range: [{submission["demand"].min():.4f}, {submission["demand"].max():.4f}]')
print(submission.head())

## 15. Ideas for Further Improvement

If you want to push past 93+:

1. **Lag features** — Since day 49 exists in both train (hours 0–2) and test (hours 2–13), you can compute rolling averages of demand for each geohash across hours within day 49 using train rows as "known history"
2. **Geohash neighbors** — Decode geohash prefix levels (5-char, 4-char) to capture neighborhood-level patterns
3. **DART boosting** — Use `boosting_type='dart'` in LGB for better generalization
4. **Neural network** — TabNet or a simple MLP as a 4th model in the stack
5. **RepeatedKFold** — Use `RepeatedKFold(n_splits=5, n_repeats=3)` for more stable OOF encoding
6. **Target encoding smoothing** — Apply additive smoothing to OOF target encoding to handle low-count categories:
   ```python
   smoothed = (count * mean + global_mean * alpha) / (count + alpha)
   ```